# Genre frequency analysis

Count how many training tracks contain each genre and compare coverage for different selections. This notebook reads `data/raw/train.csv` and writes tables and a summary to `reports/genre_analysis/`. It leaves the source data unchanged.

Run the cells from top to bottom. A track can have several genres, separated by `|`; each genre is counted once per row.

## 1. Set up paths and imports

In [1]:
from pathlib import Path
import csv
from collections import Counter

# Locate the project from either the project root or the notebooks directory.
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "data/raw/train.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from Projeto/CD or its notebooks directory.")
input_path = PROJECT_ROOT / "data/raw/train.csv"
print(f"Source: {input_path}")

import pandas as pd
from IPython.display import display
output_dir = PROJECT_ROOT / "reports/genre_analysis"


Source: /home/goncalotorcato/ist1109358/CD/Projeto/CD/data/raw/train.csv


## 2. Count genres and missing information

Frequencies count tracks, not unique artists. Tracks without genres are counted separately.

In [2]:
frequencies = Counter()
combinations = Counter()
total = 0
with input_path.open(encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    if "artist_genres" not in (reader.fieldnames or []):
        raise ValueError("Input CSV must contain an artist_genres column.")
    for row in reader:
        total += 1
        genres = frozenset(
            genre.strip()
            for genre in (row["artist_genres"] or "").split("|")
            if genre.strip()
        )
        combinations[genres] += 1
        frequencies.update(genres)

missing = combinations.pop(frozenset(), 0)
known = total - missing
if not known:
    raise ValueError("Input CSV has no tracks with known genres.")

print(f"Tracks: {total:,}; known genres: {known:,}; missing genres: {missing:,}")


Tracks: 350,051; known genres: 187,459; missing genres: 162,592


## 3. Rank genres by frequency

Ties are resolved alphabetically to keep the results reproducible. Percentages can add up to more than 100% because each track can contain several genres.

In [3]:
ranking = sorted(frequencies, key=lambda genre: (-frequencies[genre], genre))
frequency_rows = [
    {
        "rank": rank,
        "genre": genre,
        "track_count": frequencies[genre],
        "pct_all_tracks": round(100 * frequencies[genre] / total, 4),
        "pct_known_genre_tracks": round(100 * frequencies[genre] / known, 4),
    }
    for rank, genre in enumerate(ranking, start=1)
]


display(pd.DataFrame(frequency_rows).head(20))


,rank,genre,track_count,pct_all_tracks,pct_known_genre_tracks
0,1,karaoke,13491,3.8540,7.1968
1,2,rap,12524,3.5778,6.6809
2,3,pop,12240,3.4966,6.5294
3,4,rock,11691,3.3398,6.2366
4,5,hip hop,9231,2.6370,4.9243
5,6,classical,7327,2.0931,3.9086
6,7,pop rap,6908,1.9734,3.6851
7,8,dance pop,6489,1.8537,3.4616
8,9,trap,4942,1.4118,2.6363
9,10,classic rock,4661,1.3315,2.4864


## 4. Compare selection rules

Compare the top K genres with minimum-frequency thresholds. Coverage uses only tracks with known genres as the denominator. Having at least one selected genre differs from having all of a track’s genres selected.

In [4]:
selections = [
    (f"top_{k}", set(ranking[:k]))
    for k in (10, 25, 50, 100, 200, 500, 1000)
    if k <= len(ranking)
]
selections += [
    (f"min_{minimum}_tracks", {g for g in ranking if frequencies[g] >= minimum})
    for minimum in (50, 100, 250, 500, 1000)
]
selections.append(("all_genres", set(ranking)))
coverage_rows = []
for rule, selected in selections:
    covered = sum(n for genres, n in combinations.items() if genres & selected)
    fully_covered = sum(n for genres, n in combinations.items() if genres <= selected)
    coverage_rows.append({
        "rule": rule,
        "selected_genres": len(selected),
        "tracks_with_selected_genre": covered,
        "pct_known_tracks_covered": round(100 * covered / known, 4),
        "tracks_with_all_genres_selected": fully_covered,
        "pct_known_tracks_fully_covered": round(100 * fully_covered / known, 4),
    })


display(pd.DataFrame(coverage_rows))


,rule,selected_genres,tracks_with_selected_genre,pct_known_tracks_covered,tracks_with_all_genres_selected,pct_known_tracks_fully_covered
0,top_10,10,57879,30.8756,17514,9.3428
1,top_25,25,72741,38.8037,25172,13.4280
2,top_50,50,84791,45.2318,38713,20.6514
3,top_100,100,98333,52.4557,61843,32.9901
4,top_200,200,116639,62.2211,86341,46.0586
5,top_500,500,145582,77.6607,124730,66.5372
6,top_1000,1000,166281,88.7026,152535,81.3698
7,min_50_tracks,1094,168599,89.9391,156225,83.3382
8,min_100_tracks,664,154562,82.4511,136427,72.7770
9,min_250_tracks,326,133024,70.9617,106179,56.6412


## 5. Save the analysis

Export the complete ranking, coverage comparisons, and a readable summary.

In [5]:
def write_csv(path, rows, columns):
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)

output_dir.mkdir(parents=True, exist_ok=True)
write_csv(output_dir / "genre_frequencies.csv", frequency_rows, list(frequency_rows[0]))
write_csv(output_dir / "selection_coverage.csv", coverage_rows, list(coverage_rows[0]))
lines = [
    "# Genre frequency analysis", "",
    f"We analyzed `{input_path.name}` using `notebooks/01_genre_analysis.ipynb`.", "",
    f"- Track rows: {total:,}",
    f"- Tracks with known genres: {known:,}",
    f"- Tracks without genres: {missing:,} ({100 * missing / total:.2f}%)",
    f"- Distinct genres: {len(ranking):,}", "",
    "We counted each genre once per track row, so our frequencies reflect tracks rather than unique artists.",
    "A track may contribute to multiple genres, so percentages do not sum to 100%.", "",
    "## Most frequent genres", "",
    "| Genre | Tracks | % of all tracks |",
    "| --- | ---: | ---: |",
]
lines += [
    f"| {r['genre']} | {r['track_count']:,} | {r['pct_all_tracks']:.2f}% |"
    for r in frequency_rows[:20]
]
lines += [
    "", "## Selection coverage", "",
    "We calculated both coverage percentages using only tracks with known genres.",
    "At least one selected genre does not mean all of a track's genres are represented.", "",
    "| Rule | Genre columns | At least one genre covered | All genres covered |",
    "| --- | ---: | ---: | ---: |",
]
lines += [
    f"| {r['rule']} | {r['selected_genres']} | "
    f"{r['pct_known_tracks_covered']:.2f}% | {r['pct_known_tracks_fully_covered']:.2f}% |"
    for r in coverage_rows
]
top_1000 = next(row for row in coverage_rows if row["rule"] == "top_1000")
lines += [
    "", "## Initial decision", "",
    f"We selected the 1,000 most frequent genres as an initial preparation choice, "
    f"reducing the number of genre columns from {len(ranking):,} to 1,000. "
    f"Among tracks with known genres, {top_1000['pct_known_tracks_covered']:.2f}% retain "
    f"at least one genre and {top_1000['pct_known_tracks_fully_covered']:.2f}% retain all their genres. "
    "We flagged tracks without genre information with `genre_unknown`. "
    "We chose this initial trade-off between coverage and the number of features; its effect on model performance "
    "still needs to be checked using validation ROC AUC.",
]
lines += [
    "", "We will compare vocabulary sizes using validation ROC AUC before making our final selection.",
    "During model evaluation, we will calculate genre frequencies using only the training split and apply the resulting vocabulary to validation data.", "",
]
summary = "\n".join(lines)
(output_dir / "summary.md").write_text(summary, encoding="utf-8")
print(f"Reports saved to {output_dir}")


Reports saved to /home/goncalotorcato/ist1109358/CD/Projeto/CD/reports/genre_analysis


## 6. Check possible genre recovery by artist

We checked whether tracks without genres could inherit genres from other tracks with the same artist name. We also compared names together with follower counts as a consistency check. Neither field is a unique artist identifier. We used only known training genres as the reference for both datasets.

In [ ]:
import pandas as pd

artist_columns = ["artist_name", "artist_followers", "artist_genres"]
artist_train = pd.read_csv(PROJECT_ROOT / "data/raw/train.csv", usecols=artist_columns)
artist_test = pd.read_csv(PROJECT_ROOT / "data/raw/test.csv", usecols=artist_columns)

def canonical_genres(value):
    if pd.isna(value) or not value.strip():
        return None
    return "|".join(sorted({genre.strip() for genre in value.split("|") if genre.strip()}))

for frame in (artist_train, artist_test):
    frame["genre_key"] = frame["artist_genres"].map(canonical_genres)
known_artists = artist_train.dropna(subset=["genre_key"])
name_variants = known_artists.groupby("artist_name")["genre_key"].nunique()
consistent_names = name_variants[name_variants.eq(1)].index
identity_columns = ["artist_name", "artist_followers"]
pair_variants = known_artists.groupby(identity_columns)["genre_key"].nunique()
consistent_pairs = pair_variants[pair_variants.eq(1)].index
recovery_rows = []
for name, frame in (("train", artist_train), ("test", artist_test)):
    missing = frame[frame["genre_key"].isna()]
    recovery_rows.append({
        "dataset": name,
        "missing genre rows": len(missing),
        "name found with known training genres": int(missing["artist_name"].isin(name_variants.index).sum()),
        "name has one known training genre list": int(missing["artist_name"].isin(consistent_names).sum()),
        "name and follower count match consistent training metadata": int(
            pd.MultiIndex.from_frame(missing[identity_columns]).isin(consistent_pairs).sum()
        ),
    })
display(pd.DataFrame(recovery_rows))

nirvana_example = (
    artist_train[artist_train["artist_name"].eq("Nirvana")]
    .groupby(["artist_name", "artist_followers", "genre_key"], dropna=False)
    .size().rename("tracks").reset_index()
)
display(nirvana_example)


,dataset,missing genre rows,name found with known training genres,name has one known training genre list,name and follower count match consistent training metadata
0,train,162592,4953,4591,0
1,test,40733,1258,1172,0


,artist_name,artist_followers,genre_key,tracks
0,Nirvana,4370,NaN,1
1,Nirvana,20839,NaN,99
2,Nirvana,18694244,grunge|permanent wave|rock,492


### Whitespace in artist names

We also checked whether removing surrounding spaces would join rows with different follower counts. We found two affected names across train and test, each with consistent follower counts. Neither name occurs without the trailing space in the original data, so stripping spaces would standardize these names without merging existing name variants.

In [ ]:
artist_names = pd.concat([artist_train, artist_test], ignore_index=True)
artist_names["trimmed_name"] = artist_names["artist_name"].str.strip()
changed = artist_names["artist_name"].ne(artist_names["trimmed_name"])
whitespace_rows = []
for name in sorted(artist_names.loc[changed, "trimmed_name"].unique()):
    group = artist_names[artist_names["trimmed_name"].eq(name)]
    whitespace_rows.append({
        "trimmed name": name,
        "original names (repr)": ", ".join(repr(value) for value in sorted(group["artist_name"].unique())),
        "rows across train and test": len(group),
        "distinct follower counts": group["artist_followers"].nunique(),
        "follower counts": ", ".join(str(value) for value in sorted(group["artist_followers"].unique())),
        "rows already without spaces": int(group["artist_name"].eq(name).sum()),
    })
display(pd.DataFrame(whitespace_rows))


,trimmed name,original names (repr),rows across train and test,distinct follower counts,follower counts,rows already without spaces
0,Raven Black,'Raven Black ',11,1,21127,0
1,Workout Music,'Workout Music ',32,1,72649,0


### Finding

We found 4,953 training rows without genres whose artist names also occur with known genres; 4,591 of these rows match names with only one known genre list. However, no missing-genre rows in either train or test match both the name and follower count of a training artist with a consistent genre list. For example, the Nirvana rows with known genres have 18,694,244 followers, while rows without genres have 20,839 or 4,370.

These differences suggest possible artist-name collisions or inconsistent metadata. Without a unique artist identifier, we cannot establish that the rows refer to the same artist. We therefore retained `genre_unknown` and did not propagate genres by artist name.

## Interpretation and next step

Use coverage to propose vocabulary sizes, then compare their validation ROC AUC. The current preparation uses the top 1,000 genres; see `02_genre_preparation.ipynb`.

This analysis describes the full training CSV. For model evaluation, learn the frequency ranking and vocabulary from the training split only, then reuse them on validation and test data.